# Chapter 9 - Bornhuetter-Ferguson Technique

> The Bornhuetter-Ferguson technique is essentially a blend of the development
> technique and the expected claims technique.
>
> -- Friedland, Chapter 9

The Bornhuetter-Ferguson (BF) method splits ultimate claims into the claims
already reported (or paid) plus the *expected* unreported (or unpaid) claims. The
expected piece is an a priori estimate of ultimate claims (from the expected
claims technique of Chapter 8), scaled by the percentage still to emerge that is
implied by the development pattern:

$$\text{Ultimate} = \text{Actual} + \text{Expected Claims} \times \left(1 - \frac{1}{\text{CDF}}\right)$$

In the chainladder package the method is implemented by
`BornhuetterFerguson`, which takes the a priori through `sample_weight`. This
chapter recreates the **XYZ Insurer - Auto BI** example (Friedland Chapter 9),
reusing the development pattern selected for XYZ in Chapter 7.

In [1]:
import numpy as np
import pandas as pd
import chainladder as cl
import os
import tempfile
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

## The data

The XYZ Insurer Auto BI reported and paid triangles run from accident year 1998
to 2008. Their most recent diagonal (12/31/2008) is the actual claims the BF
method builds on.

In [2]:
tri = cl.load_sample("friedland_xyz_auto_bi")
reported = tri["Reported Claims"]
paid = tri["Paid Claims"]
years = list(reported.origin.year)

col = lambda t: t.to_frame(origin_as_datetime=False).iloc[:, 0].values
reported_latest = col(reported.latest_diagonal)
paid_latest = col(paid.latest_diagonal)

claims = pd.DataFrame(index=years)
claims["Reported"] = reported_latest
claims["Paid"] = paid_latest
display(claims)

,Reported,Paid
1998,15822.0,15822.0
1999,25107.0,24817.0
2000,37246.0,36782.0
2001,38798.0,38519.0
2002,48169.0,44437.0
2003,44373.0,39320.0
2004,70288.0,52811.0
2005,70655.0,40026.0
2006,48804.0,22819.0
2007,31732.0,11865.0


## Development patterns

The BF method reuses the development pattern selected for XYZ in Chapter 7: a
volume-weighted two-period average with a 1.000 reported tail and a 1.010 paid
tail. Following Friedland, the age-to-age factors are rounded to three decimals
before being cumulated to CDFs. The reported CDFs for the oldest accident years
fall just below 1.0, so they are capped at 1.0 (this avoids negative implied
unreported percentages; Friedland notes the cap is not strictly required).

In [3]:
# Reuse the Chapter 7 XYZ selection (volume-weighted two-period development
# with a constant tail). Rather than refitting, persist the fitted estimators
# as Chapter 7 would and recall them here via model persistence, keeping the
# two chapters in sync.
reported_pipe = cl.Pipeline([
    ("development", cl.Development(n_periods=2, average="volume")),
    ("tail", cl.TailConstant(tail=1.00, projection_period=0)),
]).fit(reported)
paid_pipe = cl.Pipeline([
    ("development", cl.Development(n_periods=2, average="volume")),
    ("tail", cl.TailConstant(tail=1.01, projection_period=0)),
]).fit(paid)

# Persist the fitted estimators and recall them (a model-persistence round-trip).
_pkl_dir = tempfile.gettempdir()
reported_pipe.to_pickle(os.path.join(_pkl_dir, "friedland_ch7_xyz_reported.pkl"))
paid_pipe.to_pickle(os.path.join(_pkl_dir, "friedland_ch7_xyz_paid.pkl"))
reported_pipe = cl.read_pickle(os.path.join(_pkl_dir, "friedland_ch7_xyz_reported.pkl"))
paid_pipe = cl.read_pickle(os.path.join(_pkl_dir, "friedland_ch7_xyz_paid.pkl"))

reported_dev = reported_pipe.transform(reported)
paid_dev = paid_pipe.transform(paid)

# Friedland cumulates CDFs from age-to-age factors rounded to three decimals.
reported_dev.ldf_ = reported_dev.ldf_.round(3)
paid_dev.ldf_ = paid_dev.ldf_.round(3)

# CDF to ultimate per accident year (oldest origin -> highest maturity).
reported_cdf = np.maximum(reported_dev.cdf_.to_frame(origin_as_datetime=False).values.flatten()[::-1], 1.0)
paid_cdf = paid_dev.cdf_.to_frame(origin_as_datetime=False).values.flatten()[::-1]
pct_unreported = 1 - 1 / reported_cdf
pct_unpaid = 1 - 1 / paid_cdf

patterns = pd.DataFrame(index=years)
patterns["CDF Reported"] = reported_cdf.round(3)
patterns["CDF Paid"] = paid_cdf.round(3)
patterns["% Unreported"] = pct_unreported.round(3)
patterns["% Unpaid"] = pct_unpaid.round(3)
display(patterns)

,CDF Reported,CDF Paid,% Unreported,% Unpaid
1998,1.000,1.010,0.000,0.010
1999,1.000,1.014,0.000,0.014
2000,1.000,1.031,0.000,0.030
2001,1.000,1.054,0.000,0.051
2002,1.003,1.116,0.003,0.104
2003,1.013,1.268,0.013,0.211
2004,1.064,1.525,0.060,0.344
2005,1.085,2.007,0.078,0.502
2006,1.196,3.160,0.164,0.684
2007,1.512,6.569,0.339,0.848


## Expected claims (a priori)

The a priori expected claims come from the expected claims technique (Chapter 8):
earned premium multiplied by a selected claim ratio. The earned premium is now
carried in the `friedland_xyz_auto_bi` sample and read directly from its latest
diagonal. The expected claims feed the BF method as the `sample_weight`.

In [4]:
# Earned premium is carried in the friedland_xyz_auto_bi sample ($000).
earned_premium = col(tri["Earned Premium"].latest_diagonal)

# A priori expected claims from the expected claims technique (Chapter 8, $000).
expected_claims = [15670, 24680, 35256, 39174, 47935, 54197, 86528, 108241, 70769, 39841, 39429]


def as_diagonal(tri, vec):
    """Build a per-origin (latest-diagonal) triangle from a vector of values."""
    d = tri.latest_diagonal.copy()
    d.iloc[0, 0] = np.asarray(vec, dtype=float).reshape(d.shape)
    return d


apriori = as_diagonal(reported, expected_claims)

priori = pd.DataFrame(index=years)
priori["Earned Premium"] = earned_premium
priori["Claim Ratio"] = (np.array(expected_claims) / np.array(earned_premium)).round(3)
priori["Expected Claims"] = expected_claims
display(priori)

,Earned Premium,Claim Ratio,Expected Claims
1998,20000.0,0.784,15670
1999,31500.0,0.783,24680
2000,45000.0,0.783,35256
2001,50000.0,0.783,39174
2002,61183.0,0.783,47935
2003,69175.0,0.783,54197
2004,99322.0,0.871,86528
2005,138151.0,0.783,108241
2006,107578.0,0.658,70769
2007,62438.0,0.638,39841


## Projection of ultimate claims

Applying `BornhuetterFerguson` on both the reported and paid bases produces the
projected ultimate claims. The reported IBNR is floored at zero for the capped
accident years. This recreates the *Ultimate Claims Projection* exhibit.

In [5]:
bf_reported = cl.BornhuetterFerguson(apriori=1.0).fit(reported_dev, sample_weight=apriori)
bf_paid = cl.BornhuetterFerguson(apriori=1.0).fit(paid_dev, sample_weight=apriori)

reported_ibnr = np.nan_to_num(np.maximum(col(bf_reported.ibnr_), 0.0))
reported_ult = reported_latest + reported_ibnr
paid_ult = col(bf_paid.ultimate_)

projection = pd.DataFrame(index=years)
projection["Reported"] = reported_latest
projection["Paid"] = paid_latest
projection["CDF Reported"] = reported_cdf.round(3)
projection["CDF Paid"] = paid_cdf.round(3)
projection["% Unreported"] = pct_unreported.round(3)
projection["% Unpaid"] = pct_unpaid.round(3)
projection["Earned Premium"] = earned_premium
projection["Expected Claims"] = expected_claims
projection["BF Ultimate (Reported)"] = reported_ult.round(0)
projection["BF Ultimate (Paid)"] = paid_ult.round(0)
display(projection)

,Reported,Paid,CDF Reported,CDF Paid,% Unreported,% Unpaid,Earned Premium,Expected Claims,BF Ultimate (Reported),BF Ultimate (Paid)
1998,15822.0,15822.0,1.000,1.010,0.000,0.010,20000.0,15670,15822.0,15977.0
1999,25107.0,24817.0,1.000,1.014,0.000,0.014,31500.0,24680,25107.0,25159.0
2000,37246.0,36782.0,1.000,1.031,0.000,0.030,45000.0,35256,37246.0,37851.0
2001,38798.0,38519.0,1.000,1.054,0.000,0.051,50000.0,39174,38798.0,40525.0
2002,48169.0,44437.0,1.003,1.116,0.003,0.104,61183.0,47935,48309.0,49425.0
2003,44373.0,39320.0,1.013,1.268,0.013,0.211,69175.0,54197,45066.0,50773.0
2004,70288.0,52811.0,1.064,1.525,0.060,0.344,99322.0,86528,75462.0,82612.0
2005,70655.0,40026.0,1.085,2.007,0.078,0.502,138151.0,108241,79123.0,94345.0
2006,48804.0,22819.0,1.196,3.160,0.164,0.684,107578.0,70769,60378.0,71190.0
2007,31732.0,11865.0,1.512,6.569,0.339,0.848,62438.0,39841,45229.0,45641.0


## Development of unpaid claim estimate

From the projected ultimates, the IBNR and total unpaid estimates follow by
simple differences: IBNR is ultimate minus reported claims, and total unpaid is
ultimate minus paid claims. This recreates the *Unpaid Claims* exhibit.

In [6]:
unpaid = pd.DataFrame(index=years)
unpaid["BF Ultimate (Reported)"] = reported_ult.round(0)
unpaid["BF Ultimate (Paid)"] = paid_ult.round(0)
unpaid["Case Outstanding"] = (reported_latest - paid_latest).round(0)
unpaid["IBNR (Reported)"] = (reported_ult - reported_latest).round(0)
unpaid["IBNR (Paid)"] = (paid_ult - reported_latest).round(0)
unpaid["Total Unpaid (Reported)"] = (reported_ult - paid_latest).round(0)
unpaid["Total Unpaid (Paid)"] = (paid_ult - paid_latest).round(0)
display(unpaid)

,BF Ultimate (Reported),BF Ultimate (Paid),Case Outstanding,IBNR (Reported),IBNR (Paid),Total Unpaid (Reported),Total Unpaid (Paid)
1998,15822.0,15977.0,0.0,0.0,155.0,0.0,155.0
1999,25107.0,25159.0,290.0,0.0,52.0,290.0,342.0
2000,37246.0,37851.0,464.0,0.0,605.0,464.0,1069.0
2001,38798.0,40525.0,279.0,0.0,1727.0,279.0,2006.0
2002,48309.0,49425.0,3732.0,140.0,1256.0,3872.0,4988.0
2003,45066.0,50773.0,5053.0,693.0,6400.0,5746.0,11453.0
2004,75462.0,82612.0,17477.0,5174.0,12324.0,22651.0,29801.0
2005,79123.0,94345.0,30629.0,8468.0,23690.0,39097.0,54319.0
2006,60378.0,71190.0,25985.0,11574.0,22386.0,37559.0,48371.0
2007,45229.0,45641.0,19867.0,13497.0,13909.0,33364.0,33776.0


## Reconciliation to Friedland

The selected CDFs, the projected ultimate claims, and the IBNR estimates are
reconciled to the printed Chapter 9 exhibit below (values in $000).

In [7]:
# Selected CDFs to ultimate
assert np.allclose(reported_cdf.round(3),
    [1.000, 1.000, 1.000, 1.000, 1.003, 1.013, 1.064, 1.085, 1.196, 1.512, 2.551], atol=1e-3)
assert np.allclose(paid_cdf.round(3),
    [1.010, 1.014, 1.031, 1.054, 1.116, 1.268, 1.525, 2.007, 3.160, 6.569, 21.999], atol=1e-3)
# Projected ultimate claims
assert np.allclose(reported_ult,
    [15822, 25107, 37246, 38798, 48309, 45066, 75462, 79123, 60378, 45229, 42607], atol=1)
assert np.allclose(paid_ult,
    [15977, 25159, 37851, 40525, 49425, 50773, 82612, 94345, 71190, 45641, 41046], atol=1)
# IBNR (ultimate minus reported)
assert np.allclose(unpaid["IBNR (Reported)"].values,
    [0, 0, 0, 0, 140, 693, 5174, 8468, 11574, 13497, 23975], atol=1)
assert np.allclose(unpaid["IBNR (Paid)"].values,
    [155, 52, 605, 1727, 1256, 6400, 12324, 23690, 22386, 13909, 22414], atol=1)